In [168]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

In [169]:
def load_pdfs(pdf_dir: str = "../data/pdf") -> list:
    """Load all PDF files and enrich each document with source metadata."""
    pdf_paths = sorted(Path(pdf_dir).glob("*.pdf"))

    if not pdf_paths:
        print(f"No PDF files found in '{pdf_dir}'")
        return []

    documents = []
    for path in pdf_paths:
        docs = PyPDFLoader(str(path)).load()
        for doc in docs:
            doc.metadata.update({
                "source":      str(path),
                "file_name":   path.name,
                "file_type":   path.suffix.lower(),   # ".pdf"
                "file_size_kb": round(path.stat().st_size / 1024, 2),
            })
        documents.extend(docs)
        print(f"Loaded '{path.name}' — {len(docs)} page(s)")

    print(f"\nTotal pages: {len(documents)}")
    return documents

In [170]:
docs = load_pdfs()

Loaded 'Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf' — 6 page(s)
Loaded 'SeveScents-Invoice-ORD-20260607-GCME8Q.pdf' — 1 page(s)

Total pages: 7


In [171]:
# Inspect metadata of the first document
if docs:
    print(docs[0].metadata)

{'producer': 'Microsoft® Excel® 2019', 'creator': 'Microsoft® Excel® 2019', 'creationdate': '2026-04-23T16:09:50+05:00', 'author': 'openpyxl', 'moddate': '2026-04-23T16:09:50+05:00', 'source': '..\\data\\pdf\\Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'file_name': 'Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf', 'file_type': '.pdf', 'file_size_kb': 570.03}


In [172]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents: list, chunk_size: int = 1000, chunk_overlap: int = 200) -> list:
    """Split documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"Split {len(documents)} page(s) → {len(chunks)} chunk(s)  "
          f"(size={chunk_size}, overlap={chunk_overlap})")
    return chunks

In [173]:
chunks = split_documents(docs)

# Inspect first chunk
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

Split 7 page(s) → 19 chunk(s)  (size=1000, overlap=200)
{'producer': 'Microsoft® Excel® 2019', 'creator': 'Microsoft® Excel® 2019', 'creationdate': '2026-04-23T16:09:50+05:00', 'author': 'openpyxl', 'moddate': '2026-04-23T16:09:50+05:00', 'source': '..\\data\\pdf\\Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'file_name': 'Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf', 'file_type': '.pdf', 'file_size_kb': 570.03}
Sr. No. Variation Of…  100گرام ریٹفی کلوگرام ریٹ خوشبو کا نامسیریل نمیر 
1 1 MILLION (PACO RABANNE) 2580 22800ون ملی ن (پیکو ربانن)1
2 1 MILLION ROYAL (PACO RABANNE) - PRM 5300 50000ون ملی ن رائل (پیکو ربانن) - پریمیم2
3 1872 FOR MEN (CLIVE CHRISTIAN) - PRM 5300 50000  1872فار می ن (کلائیو کرسچی ن )


In [174]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv("../.env")  # loads OPENAI_API_KEY into os.environ

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — copy .env.example → .env and fill in your key"
print("API key loaded.")

API key loaded.


In [175]:
class Embedder:
    """Wraps OpenAI text-embedding-3-small for RAG indexing and retrieval."""

    def __init__(self, model: str = "text-embedding-3-small"):
        self.model = model
        self._embeddings = OpenAIEmbeddings(model=model)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed a batch of texts (used when building the vector store)."""
        return self._embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed a single query string (used at retrieval time)."""
        return self._embeddings.embed_query(text)

    def as_langchain(self) -> OpenAIEmbeddings:
        """Return the raw LangChain embeddings object (for vector store constructors)."""
        return self._embeddings

In [176]:
embedder = Embedder()

# Smoke-test: embed the first chunk
vector = embedder.embed_query(chunks[0].page_content)
print(f"Model  : {embedder.model}")
print(f"Dim    : {len(vector)}")
print(f"Sample : {vector[:5]}")

Model  : text-embedding-3-small
Dim    : 1536
Sample : [0.036376953125, 0.026123046875, 0.02459716796875, -0.0257568359375, -0.030242919921875]


In [177]:
import uuid
import numpy as np
import chromadb
from pathlib import Path


class VectorStore:
    """Manages document embeddings in a persistent ChromaDB collection."""

    def __init__(self, persist_dir: str = "../data/chroma", collection_name: str = "rag_docs"):
        self.persist_dir     = Path(persist_dir)
        self.collection_name = collection_name
        self.persist_dir.mkdir(parents=True, exist_ok=True)

        self.client     = chromadb.PersistentClient(path=str(self.persist_dir))
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )
        print(f"Collection '{collection_name}' ready — {self.collection.count()} doc(s) on disk")

    def reset(self) -> None:
        """Drop and recreate the collection (clears all stored documents)."""
        self.client.delete_collection(self.collection_name)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"},
        )
        print(f"Collection '{self.collection_name}' reset — 0 doc(s)")

    def add_documents(self, documents: list, embeddings: list | np.ndarray) -> None:
        """Insert documents + their embeddings into ChromaDB."""
        if len(documents) != len(embeddings):
            raise ValueError(
                f"Mismatch: {len(documents)} document(s) but {len(embeddings)} embedding(s)."
            )

        ids, vecs, metas, texts = [], [], [], []

        for idx, (doc, emb) in enumerate(zip(documents, embeddings)):
            ids.append(str(uuid.uuid4()))
            vecs.append(emb.tolist() if isinstance(emb, np.ndarray) else list(emb))
            meta = {**doc.metadata, "chunk_index": idx, "content_length": len(doc.page_content)}
            meta = {k: v for k, v in meta.items() if isinstance(v, (str, int, float, bool))}
            metas.append(meta)
            texts.append(doc.page_content)

        self.collection.add(ids=ids, embeddings=vecs, metadatas=metas, documents=texts)
        print(f"Added {len(ids)} doc(s) — collection total: {self.collection.count()}")

    def query(self, query_text: str, embedder, top_k: int = 5) -> list[dict]:
        """Convert query_text to an embedding, run cosine search, return top_k results."""
        query_vector = embedder.embed_query(query_text)

        results = self.collection.query(
            query_embeddings=[query_vector],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )

        hits = []
        for rank, (text, meta, distance) in enumerate(
            zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
            start=1,
        ):
            hits.append({
                "rank":     rank,
                "score":    round(1 - distance, 4),
                "text":     text,
                "metadata": meta,
            })

        return hits

In [ ]:
vs = VectorStore()

if vs.collection.count() == 0:
    print("Empty collection — generating embeddings and indexing documents...")
    chunk_texts      = [c.page_content for c in chunks]
    chunk_embeddings = embedder.embed_documents(chunk_texts)
    vs.add_documents(chunks, chunk_embeddings)
else:
    print(f"Collection already has {vs.collection.count()} doc(s) — skipping embedding.")

In [179]:
results = vs.query("1 Million perfume?", embedder, top_k=3)

for hit in results:
    print(f"[{hit['rank']}] score={hit['score']}  file={hit['metadata'].get('file_name')}")
    print(f"    {hit['text'][:200]}\n")

[1] score=0.536  file=Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf
    Sr. No. Variation Of…  100گرام ریٹفی کلوگرام ریٹ خوشبو کا نامسیریل نمیر 
1 1 MILLION (PACO RABANNE) 2580 22800ون ملی ن (پیکو ربانن)1
2 1 MILLION ROYAL (PACO RABANNE) - PRM 5300 50000ون ملی ن رائل (پیک

[2] score=0.5336  file=Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf
    98 INVICTUS VICTORY ABSOLU (PACO RABANNE) 3000 27000انوکٹس وکیری ابسولو (پیکو ربانن)98
99 J'ADORE (DIOR) 2420 21200جاڈور(ڈیور)99
100 JIMMY CHOO ICE (JIMMY CHOO) 2750 24500جمی چو آئس(جمی چو)100
101 JIM

[3] score=0.5056  file=Argeville - Fine Fragrances - Price List (KG & 100 GRAMS).pdf
    86 HACIVAT (NISHANE) 3900 36000حاجیوات(نشانن)86
87 HALFETI (PENHALIGON'S) 3200 29000ہالفیٹی(پی ن ہالیگینن )87
88 HALTANE (PARFUMS DE MARLY) - PRM 6550 62500ہالٹی ن (پارفمز ڈی مارلی) - پریمیم88
89 HIBI



In [180]:
from langchain_openai import ChatOpenAI


class RAG:
    """Retrieval-Augmented Generation: fetch relevant chunks then answer with an LLM."""

    PROMPT_TEMPLATE = """\
You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}
Answer:"""

    def __init__(self, vector_store: VectorStore, embedder: Embedder,
                 model: str = "gpt-5.4-nano", top_k: int = 5):
        self.vector_store = vector_store
        self.embedder     = embedder
        self.top_k        = top_k
        self.llm          = ChatOpenAI(model=model, temperature=0)

    def answer(self, question: str) -> str:
        """Retrieve context for question, then generate and return an answer."""
        hits = self.vector_store.query(question, self.embedder, top_k=self.top_k)

        context = "\n\n---\n\n".join(
            f"[Source: {h['metadata'].get('file_name', 'unknown')}  "
            f"page {h['metadata'].get('page', '?')}  score={h['score']}]\n{h['text']}"
            for h in hits
        )

        prompt  = self.PROMPT_TEMPLATE.format(context=context, question=question)
        response = self.llm.invoke(prompt)
        return response.content

In [181]:
rag = RAG(vs, embedder)

print(rag.answer("1 million perfume?"))

**1 MILLION (PACO RABANNE)** — **100 grams: 22800** and **1 kg: 2580**.
